# Congruences of the prime-detecting quasimodular forms $H_k$

Companion notebook to Section 4.2 and Appendix A of the bachelor's thesis
*Prime-Detecting Quasimodular Forms and Some of Their Congruence Properties*
(Moh Hasbi Alfarizi, Universitas Gadjah Mada, 2026).

The notebook builds the $q$-expansions of the Eisenstein series $G_k$ and of the prime-detecting
quasimodular forms $H_k$, checks their basic properties, and searches for congruences

$$a(tn+b) \equiv 0 \pmod m \quad \text{for all } n \ge 0, \qquad a(n) = b_n(H_k).$$

1. Eisenstein series and the operator $D$
2. The forms $H_k$
3. The congruence search of Appendix A
4. Refining the search
5. Checking the results of Section 4.2
6. Other weights

**Requirements:** SageMath (tested with 10.2). The whole notebook runs in about 20 seconds.

## 1. Eisenstein series and the operator D

For even $k \ge 2$, the normalized Eisenstein series (Equation 3.3) is

$$G_k = -\frac{B_k}{2k} + \sum_{n \ge 1} \sigma_{k-1}(n)\, q^n, \qquad \sigma_j(n) = \sum_{d \mid n} d^j,$$

where $B_k$ is the $k$-th Bernoulli number. The operator $D = q\frac{d}{dq} = \frac{1}{2\pi i}\frac{d}{dz}$
multiplies the coefficient of $q^n$ by $n$, so $D^d$ multiplies it by $n^d$ and removes the constant term.

`Gfunct(k, N, d)` returns $D^d G_k$ truncated after $q^N$. It works in Sage's power-series ring over
$\mathbb{Q}$, so the coefficients are exact rationals.

In [1]:
R.<q> = PowerSeriesRing(QQ)

def Gfunct(k, N, d=0):
    # D^d G_k truncated after q^N, where D = q d/dq
    if k < 2 or k % 2:
        raise ValueError("k must be an even integer >= 2")
    # D kills the constant term -B_k/(2k) and multiplies the coefficient of q^n by n
    coeffs = [-bernoulli(k) / (2*k) if d == 0 else 0]
    coeffs += [sigma(n, k - 1) * n^d for n in range(1, N + 1)]
    return R(coeffs).add_bigoh(N + 1)

print("G_4      =", Gfunct(4, 8))
print("D^2 G_4  =", Gfunct(4, 8, 2))

G_4      = 1/240 + q + 9*q^2 + 28*q^3 + 73*q^4 + 126*q^5 + 252*q^6 + 344*q^7 + 585*q^8 + O(q^9)
D^2 G_4  = q + 36*q^2 + 252*q^3 + 1168*q^4 + 3150*q^5 + 9072*q^6 + 16856*q^7 + 37440*q^8 + O(q^9)


In [2]:
# Cross-check against Sage's built-in q-expansion, which uses the same normalization
assert all(Gfunct(k, 200) == eisenstein_series_qexp(k, 201) for k in range(2, 32, 2))
print("Gfunct agrees with eisenstein_series_qexp for k = 2, 4, ..., 30.")

Gfunct agrees with eisenstein_series_qexp for k = 2, 4, ..., 30.


## 2. The forms H_k

For even $k \ge 6$ (Definition 4.2.1),

$$H_k = \sum_{n \ge 0} b_n(H_k)\, q^n =
\begin{cases}
\frac{1}{6}\left((D^2 - D + 1)G_2 - G_4\right), & k = 6,\\
\frac{1}{24}\left(-D^2 G_{k-6} + (D^2 + 1)G_{k-4} - G_{k-2}\right), & k \ge 8.
\end{cases}$$

Reading off the coefficients gives, for $n \ge 1$,

$$b_n(H_6) = \frac{(n^2 - n + 1)\,\sigma_1(n) - \sigma_3(n)}{6}, \qquad
b_n(H_k) = \frac{-n^2 \sigma_{k-7}(n) + (n^2 + 1)\,\sigma_{k-5}(n) - \sigma_{k-3}(n)}{24} \quad (k \ge 8).$$

The forms are prime-detecting: for $n \ge 2$, $b_n(H_k) = 0$ exactly when $n$ is prime
(Craig, van Ittersum and Ono, 2024).

In [3]:
def Hfunct(k, N):
    # The prime-detecting quasimodular form H_k truncated after q^N (Definition 4.2.1)
    if k < 6 or k % 2:
        raise ValueError("H_k is defined for even k >= 6")
    G = lambda w, d=0: Gfunct(w, N, d)
    if k == 6:
        return (G(2, 2) - G(2, 1) + G(2) - G(4)) / 6
    return (-G(k - 6, 2) + G(k - 4, 2) + G(k - 4) - G(k - 2)) / 24

def b_closed(k, n):
    # b_n(H_k) for n >= 1 from the divisor-sum formula
    if k == 6:
        return ((n^2 - n + 1)*sigma(n, 1) - sigma(n, 3)) / 6
    return (-n^2*sigma(n, k - 7) + (n^2 + 1)*sigma(n, k - 5) - sigma(n, k - 3)) / 24

def get_coefficients(series, N):
    # [b_0, b_1, ..., b_N]; b_0 stays rational, b_n for n >= 1 are integers (Theorem 4.2.5)
    coeffs = series.padded_list(N + 1)
    assert all(c in ZZ for c in coeffs[1:]), "non-integral coefficient"
    return [coeffs[0]] + [ZZ(c) for c in coeffs[1:]]

print("H_6 =", Hfunct(6, 16))
print("H_8 =", Hfunct(8, 16))

H_6 = -11/1440 + 3*q^4 + 20*q^6 + 45*q^8 + 32*q^9 + 84*q^10 + 280*q^12 + 216*q^14 + 256*q^15 + 465*q^16 + O(q^17)
H_8 = 31/120960 + 3*q^4 + 35*q^6 + 135*q^8 + 72*q^9 + 399*q^10 + 1435*q^12 + 2106*q^14 + 1216*q^15 + 4695*q^16 + O(q^17)


The constant term $b_0(H_k)$ is not an integer (for $H_6$ it is $-11/1440$), so it is kept as a
rational number and left out of the congruence search. Every other coefficient is an integer, and
$b_1(H_k) = 0$.

In [4]:
N = 2000
for k in range(6, 32, 2):
    b = get_coefficients(Hfunct(k, N), N)
    assert b[1] == 0
    assert all(b[n] == b_closed(k, n) for n in range(1, N + 1))
    assert all((b[n] == 0) == is_prime(n) for n in range(2, N + 1))

print(f"For k = 6, 8, ..., 30 and n <= {N}: the coefficients b_n(H_k) are integers, match the")
print("divisor-sum formula, and vanish exactly when n is prime.")

For k = 6, 8, ..., 30 and n <= 2000: the coefficients b_n(H_k) are integers, match the
divisor-sum formula, and vanish exactly when n is prime.


## 3. The congruence search of Appendix A

For each modulus $m$, step $t$ and offset $0 \le b < t$, the search checks every coefficient
$a(b), a(b+t), a(b+2t), \dots$ up to $a(N)$ and stops at the first one not divisible by $m$.
If none fails, it records $a(tn+b) \equiv 0 \pmod m$. Progressions start at index $1$
(for $b = 0$ the first index is $t$) because $b_0(H_k)$ is not an integer.

The cell below runs the search with the parameters of Appendix A ($N = 100$, $m \le 13$, $t \le 12$).
Its output modulo $2$ and $3$ corresponds to Figures A.4.1 and A.4.2.

In [5]:
def progression(t, b, N):
    # Indices 1 <= i <= N with i = b (mod t)
    return range(b if b > 0 else t, N + 1, t)

def find_congruences(coeffs, moduli, max_t):
    # All (m, t, b) such that m divides coeffs[i] for every i in progression(t, b, N)
    N = len(coeffs) - 1
    found = []
    for m in moduli:
        for t in range(1, max_t + 1):
            for b in range(t):
                if all(coeffs[i] % m == 0 for i in progression(t, b, N)):
                    found.append((m, t, b))
    return found

NUM_COEFFS = 100
MAX_MODULUS_TO_CHECK = 13
MAX_T_TO_CHECK = 12

coeffs = get_coefficients(Hfunct(6, NUM_COEFFS), NUM_COEFFS)
raw = find_congruences(coeffs, range(2, MAX_MODULUS_TO_CHECK + 1), MAX_T_TO_CHECK)

print(f"{len(raw)} congruences for H_6 up to q^{NUM_COEFFS}. Those modulo 2 and 3:\n")
for m, t, b in raw:
    if m in (2, 3):
        print(f"a({t}n + {b}) ≡ 0 (mod {m})")

154 congruences for H_6 up to q^100. Those modulo 2 and 3:

a(2n + 1) ≡ 0 (mod 2)
a(4n + 1) ≡ 0 (mod 2)
a(4n + 2) ≡ 0 (mod 2)
a(4n + 3) ≡ 0 (mod 2)
a(6n + 1) ≡ 0 (mod 2)
a(6n + 3) ≡ 0 (mod 2)
a(6n + 5) ≡ 0 (mod 2)
a(7n + 0) ≡ 0 (mod 2)
a(7n + 3) ≡ 0 (mod 2)
a(7n + 5) ≡ 0 (mod 2)
a(7n + 6) ≡ 0 (mod 2)
a(8n + 1) ≡ 0 (mod 2)
a(8n + 2) ≡ 0 (mod 2)
a(8n + 3) ≡ 0 (mod 2)
a(8n + 5) ≡ 0 (mod 2)
a(8n + 6) ≡ 0 (mod 2)
a(8n + 7) ≡ 0 (mod 2)
a(9n + 2) ≡ 0 (mod 2)
a(9n + 3) ≡ 0 (mod 2)
a(9n + 6) ≡ 0 (mod 2)
a(10n + 1) ≡ 0 (mod 2)
a(10n + 3) ≡ 0 (mod 2)
a(10n + 5) ≡ 0 (mod 2)
a(10n + 7) ≡ 0 (mod 2)
a(10n + 9) ≡ 0 (mod 2)
a(11n + 0) ≡ 0 (mod 2)
a(11n + 2) ≡ 0 (mod 2)
a(11n + 7) ≡ 0 (mod 2)
a(12n + 1) ≡ 0 (mod 2)
a(12n + 2) ≡ 0 (mod 2)
a(12n + 3) ≡ 0 (mod 2)
a(12n + 5) ≡ 0 (mod 2)
a(12n + 6) ≡ 0 (mod 2)
a(12n + 7) ≡ 0 (mod 2)
a(12n + 9) ≡ 0 (mod 2)
a(12n + 10) ≡ 0 (mod 2)
a(12n + 11) ≡ 0 (mod 2)
a(3n + 1) ≡ 0 (mod 3)
a(3n + 2) ≡ 0 (mod 3)
a(6n + 1) ≡ 0 (mod 3)
a(6n + 2) ≡ 0 (mod 3)
a(6n + 4) ≡ 0 (mod 

## 4. Refining the search

The raw list has two problems.

1. **Most entries are redundant.** $a(4n+1)$, $a(4n+3)$, $a(6n+5)$, ... are all sub-progressions of
   $a(2n+1) \equiv 0 \pmod 2$. Likewise, a congruence modulo $6$ is the same as one modulo $2$ together
   with one modulo $3$, so only prime-power moduli carry new information.
2. **Some entries are artifacts of the small range.** With $N = 100$, a progression with step $11$
   contains only nine coefficients, and several of them are zero just because their index is prime.
   Checking more coefficients removes these entries.

In [6]:
def first_failure(coeffs, m, t, b):
    # Smallest index in the progression whose coefficient is not divisible by m, or None
    N = len(coeffs) - 1
    return next((i for i in progression(t, b, N) if coeffs[i] % m), None)

N_BIG = 10000
coeffs_big = get_coefficients(Hfunct(6, N_BIG), N_BIG)

print(f"Entries of the raw list that fail when the search goes up to q^{N_BIG}:\n")
for m, t, b in raw:
    i = first_failure(coeffs_big, m, t, b)
    if i is not None:
        print(f"a({t}n + {b}) ≡ 0 (mod {m})  fails at index {i}: b_{i}(H_6) = {coeffs_big[i]}")

Entries of the raw list that fail when the search goes up to q^10000:

a(7n + 0) ≡ 0 (mod 2)  fails at index 196: b_196(H_6) = 1106115
a(9n + 2) ≡ 0 (mod 2)  fails at index 128: b_128(H_6) = 291465
a(11n + 0) ≡ 0 (mod 2)  fails at index 484: b_484(H_6) = 14703579
a(11n + 2) ≡ 0 (mod 2)  fails at index 200: b_200(H_6) = 1548855
a(11n + 7) ≡ 0 (mod 2)  fails at index 128: b_128(H_6) = 291465
a(7n + 0) ≡ 0 (mod 4)  fails at index 196: b_196(H_6) = 1106115
a(11n + 0) ≡ 0 (mod 4)  fails at index 484: b_484(H_6) = 14703579
a(12n + 7) ≡ 0 (mod 5)  fails at index 115: b_115(H_6) = 59136
a(9n + 2) ≡ 0 (mod 6)  fails at index 128: b_128(H_6) = 291465
a(7n + 0) ≡ 0 (mod 8)  fails at index 196: b_196(H_6) = 1106115
a(12n + 7) ≡ 0 (mod 10)  fails at index 115: b_115(H_6) = 59136


`find_primitive_congruences` fixes both problems. It checks prime-power moduli only, skips
progressions with fewer than `min_terms` coefficients, and reports a progression only if it is not
already covered by congruences with a smaller step for the same modulus. It then prints the result
grouped by step, e.g. `n ≡ 1, 2 (mod 3)` for $a(3n+1) \equiv a(3n+2) \equiv 0$.

In [7]:
def find_primitive_congruences(coeffs, max_mod, max_t, min_terms=50):
    # {m: [(t, b), ...]} for prime powers m <= max_mod, without the progressions that are
    # covered by congruences with a smaller step for the same m
    N = len(coeffs) - 1
    result = {}
    for m in filter(is_prime_power, range(2, max_mod + 1)):
        covered = [False] * (N + 1)
        result[m] = []
        for t in range(1, max_t + 1):
            for b in range(t):
                idx = progression(t, b, N)
                if len(idx) < min_terms or all(covered[i] for i in idx):
                    continue
                if all(coeffs[i] % m == 0 for i in idx):
                    result[m].append((t, b))
                    for i in idx:
                        covered[i] = True
    return result

def show_primitive(k, result):
    print(f"H_{k}:")
    for m, progs in result.items():
        if not progs:
            continue
        steps = sorted(set(t for t, _ in progs))
        groups = [", ".join(str(b) for s, b in progs if s == t) + f" (mod {t})" for t in steps]
        print(f"  a(n) ≡ 0 (mod {m:>2})  for  n ≡ " + "  or  n ≡ ".join(groups))

show_primitive(6, find_primitive_congruences(coeffs_big, MAX_MODULUS_TO_CHECK, MAX_T_TO_CHECK))

H_6:
  a(n) ≡ 0 (mod  2)  for  n ≡ 1 (mod 2)  or  n ≡ 2 (mod 4)  or  n ≡ 3, 5, 6 (mod 7)  or  n ≡ 3, 6 (mod 9)
  a(n) ≡ 0 (mod  3)  for  n ≡ 1, 2 (mod 3)
  a(n) ≡ 0 (mod  4)  for  n ≡ 1 (mod 2)  or  n ≡ 2 (mod 4)  or  n ≡ 3, 6 (mod 9)
  a(n) ≡ 0 (mod  5)  for  n ≡ 1, 2, 3 (mod 5)
  a(n) ≡ 0 (mod  7)  for  n ≡ 3, 5 (mod 7)
  a(n) ≡ 0 (mod  8)  for  n ≡ 1 (mod 2)  or  n ≡ 3 (mod 9)
  a(n) ≡ 0 (mod  9)  for  n ≡ 2 (mod 3)


The 154 raw entries reduce to a short list, and every entry holds for all $n \le 10^4$.
The results proved in the thesis appear as $a(2n+1) \equiv 0 \pmod 2$ (Theorem 4.2.11) and
$a(3n+1) \equiv a(3n+2) \equiv 0 \pmod 3$ (Theorem 4.2.12). The other entries, such as
$a(2n+1) \equiv 0 \pmod 8$, $a(3n+2) \equiv 0 \pmod 9$ and the congruences modulo $5$ and $7$,
are numerical observations: they are not proved in the thesis.

## 5. Checking the results of Section 4.2

With $S_6(n) = 6\,b_n(H_6)$ and $S_k(n) = 24\,b_n(H_k)$ for $k \ge 8$, the thesis proves:

| Result | Statement |
|---|---|
| Theorem 4.2.5 | $b_n(H_k) \in \mathbb{Z}$ for $n \ge 2$ |
| Theorem 4.2.9 | $S_6(n) \equiv 0 \pmod 9$ if $n \not\equiv 0 \pmod 3$ |
| Lemma 4.2.10 | $S_k(n) \equiv 0 \pmod{16}$ if $n$ is odd ($k \ge 8$) |
| Theorem 4.2.11 | $b_n(H_k) \equiv 0 \pmod 2$ if $n$ is odd |
| Theorem 4.2.12 | $b_n(H_k) \equiv 0 \pmod 3$ if $n \not\equiv 0 \pmod 3$ |

The cell below checks all five for every even $k$ from $6$ to $40$ and every $n \le 10^4$.

In [8]:
N = 10000
for k in range(6, 42, 2):
    b = get_coefficients(Hfunct(k, N), N)     # asserts Theorem 4.2.5
    S = [(6 if k == 6 else 24) * c for c in b]
    if k == 6:
        assert all(S[n] % 9 == 0 for n in range(1, N + 1) if n % 3)          # Theorem 4.2.9
    else:
        assert all(S[n] % 16 == 0 for n in range(1, N + 1, 2))               # Lemma 4.2.10
    assert all(b[n] % 2 == 0 for n in range(1, N + 1, 2))                    # Theorem 4.2.11
    assert all(b[n] % 3 == 0 for n in range(1, N + 1) if n % 3)              # Theorem 4.2.12

print(f"Theorems 4.2.5, 4.2.9, 4.2.11, 4.2.12 and Lemma 4.2.10 hold for k = 6, ..., 40 and n <= {N}.")

Theorems 4.2.5, 4.2.9, 4.2.11, 4.2.12 and Lemma 4.2.10 hold for k = 6, ..., 40 and n <= 10000.


## 6. Other weights

Section 5.2 of the thesis suggests looking for congruences modulo larger primes. The cell below runs
the refined search for $k = 6, 8, \dots, 40$ with $N = 10^4$, prime powers $m \le 25$ and steps
$t \le 23$. It prints $H_6$ and $H_8$ in full and, for the other weights, only the moduli whose
congruences differ from those of $H_8$.

In [9]:
N = 10000
results = {}
for k in range(6, 42, 2):
    coeffs = get_coefficients(Hfunct(k, N), N)
    results[k] = find_primitive_congruences(coeffs, max_mod=25, max_t=23)

show_primitive(6, results[6])
show_primitive(8, results[8])
print("\nModuli where H_k differs from H_8:")
for k in range(10, 42, 2):
    diff = {m: progs for m, progs in results[k].items() if progs != results[8][m]}
    if diff:
        show_primitive(k, diff)
    else:
        print(f"H_{k}: same as H_8")

H_6:
  a(n) ≡ 0 (mod  2)  for  n ≡ 1 (mod 2)  or  n ≡ 2 (mod 4)  or  n ≡ 3, 5, 6 (mod 7)  or  n ≡ 3, 6 (mod 9)  or  n ≡ 7, 11, 13, 14 (mod 15)  or  n ≡ 12 (mod 16)  or  n ≡ 3, 5, 6, 7, 10, 11, 12, 14 (mod 17)  or  n ≡ 5, 7, 10, 11, 14, 15, 17, 19, 20, 21, 22 (mod 23)
  a(n) ≡ 0 (mod  3)  for  n ≡ 1, 2 (mod 3)
  a(n) ≡ 0 (mod  4)  for  n ≡ 1 (mod 2)  or  n ≡ 2 (mod 4)  or  n ≡ 3, 6 (mod 9)  or  n ≡ 12 (mod 16)
  a(n) ≡ 0 (mod  5)  for  n ≡ 1, 2, 3 (mod 5)  or  n ≡ 8 (mod 16)
  a(n) ≡ 0 (mod  7)  for  n ≡ 3, 5 (mod 7)
  a(n) ≡ 0 (mod  8)  for  n ≡ 1 (mod 2)  or  n ≡ 3 (mod 9)  or  n ≡ 12, 14 (mod 16)
  a(n) ≡ 0 (mod  9)  for  n ≡ 2 (mod 3)
  a(n) ≡ 0 (mod 16)  for  n ≡ 1 (mod 2)
H_8:
  a(n) ≡ 0 (mod  2)  for  n ≡ 1 (mod 2)  or  n ≡ 14 (mod 16)
  a(n) ≡ 0 (mod  3)  for  n ≡ 1, 2 (mod 3)
  a(n) ≡ 0 (mod  4)  for  n ≡ 1 (mod 2)
  a(n) ≡ 0 (mod  5)  for  n ≡ 1, 2, 3 (mod 5)  or  n ≡ 8 (mod 16)
  a(n) ≡ 0 (mod  7)  for  n ≡ 3, 5, 6 (mod 7)
  a(n) ≡ 0 (mod  8)  for  n ≡ 1 (mod 2)
  a(n) ≡ 0 (m

Some patterns in this output, all of them **numerical observations for $n \le 10^4$, not proved here**:

* For every weight: $a(2n+1) \equiv 0 \pmod 8$, $a(3n+2) \equiv 0 \pmod 9$,
  $a(n) \equiv 0 \pmod 5$ for $n \equiv 1, 2, 3 \pmod 5$, and $a(n) \equiv 0 \pmod 7$ for $n \equiv 3, 5 \pmod 7$.
* All weights $k \ge 8$ share the same congruences modulo $2, 3, 4, 5, 8, 9$ and $16$.
* Some congruences appear only for certain weights. Modulo $11$ they appear for $k \equiv 0 \pmod{10}$,
  modulo $19$ for $k \equiv 14 \pmod{18}$, and modulo $23$ for $k \equiv 16 \pmod{22}$.
  Modulo $7$, the extra class $n \equiv 6$ appears for $k \equiv 2 \pmod 6$. In each case the residue
  classes are exactly the quadratic non-residues modulo the prime.
* For $k \equiv 0 \pmod{10}$ there is also $a(5n+1) \equiv 0 \pmod{25}$.

Proving or explaining these is left open.